# Exp. 1 Baseline: Bicubic Upsampling

A super-resolution style baseline for Exp. 1 (political campaigning).
The regional outcomes are treated as a coarse $10 \times 10$ image and upsampled
to the $40 \times 40$ subregion grid with bicubic interpolation. The interpolated map is then
used as a pseudo high-resolution training target for the same network Clam uses
(exactly like the uniform baseline, but with a smooth target instead of a constant one).

**Expectation.** Bicubic upsampling only exploits *where* a subregion is, never its context $c$.
Since the heterogeneity in Exp. 1 is context-driven and spatially unstructured, the baseline
should not recover the local causal effects (LOCATE).

## Setup and data (identical to Exp. 1)

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

# Parameters, identical to Exp. 1 (political campaigning)
SEED = 42
ROW = 100          # number of regions, arranged on a 10 x 10 grid
COL = 16           # subregions per region, arranged on a 4 x 4 grid
N_SWAPS = 100      # random pair swaps in the context matrix
NOISE_STD = 0.1    # observation noise on the subregional outcomes
N_EPOCHS = 4000
LR = 1e-3
WEIGHT_DECAY = 1e-3
HIDDEN_DIM = 32
DROPOUT = 0.05


def set_all_seeds(seed):
    np.random.seed(seed)
    torch.manual_seed(seed)


In [ ]:
def true_f(t, c):
    """Ground-truth mechanism: outcome is 1 if untreated,
    and a piecewise-linear function of the context if treated."""
    anchor_x = np.array([-100, -2, 0, 1, 100], dtype=float)
    anchor_y = np.array([2, 2, 1, 0, 0], dtype=float)
    return np.where(t == 0, 1.0, np.interp(c, anchor_x, anchor_y))


def create_context_matrix():
    """Row-centered Gaussian context with random swaps,
    so the regional mean context is uninformative by construction."""
    C = np.random.normal(size=(ROW, COL))
    C = C - C.mean(axis=1, keepdims=True)
    for _ in range(N_SWAPS):
        r1, c1 = np.random.randint(ROW), np.random.randint(COL)
        r2, c2 = np.random.randint(ROW), np.random.randint(COL)
        C[r1, c1], C[r2, c2] = C[r2, c2], C[r1, c1]
    return C


def create_treatment_matrix():
    """Half of the regions are treated; treatment covers all subregions of a region."""
    flags = np.array([1] * (ROW // 2) + [0] * (ROW // 2))
    np.random.shuffle(flags)
    return np.repeat(flags[:, None], COL, axis=1).astype(float)


set_all_seeds(SEED)
T = create_treatment_matrix()
C = create_context_matrix()
Y = true_f(T, C) + np.random.normal(scale=NOISE_STD, size=(ROW, COL))

Y_agg = Y.mean(axis=1)                                # observed regional outcomes (the only supervision)
E_true = true_f(T, C) - true_f(np.zeros_like(T), C)   # ground-truth LOCATE, used only for evaluation


## Model and training (identical to Exp. 1)

In [ ]:
class SmallMLP(nn.Module):
    """f_theta(t, c): two inputs, one scalar output. Same architecture as in Exp. 1."""

    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2, HIDDEN_DIM), nn.ReLU(), nn.Dropout(DROPOUT),
            nn.Linear(HIDDEN_DIM, HIDDEN_DIM), nn.ReLU(), nn.Dropout(DROPOUT),
            nn.Linear(HIDDEN_DIM, 1),
        )

    def forward(self, x):
        return self.net(x).squeeze(-1)


def features(T_mat, C_mat):
    return torch.tensor(np.stack([T_mat, C_mat], axis=-1), dtype=torch.float32).reshape(-1, 2)


def locate_matrix(model, T_mat, C_mat):
    """LOCATE estimate: f(t, c) - f(0, c) for every subregion."""
    model.eval()
    with torch.no_grad():
        pred = model(features(T_mat, C_mat)).reshape(ROW, COL)
        pred0 = model(features(np.zeros_like(T_mat), C_mat)).reshape(ROW, COL)
    return (pred - pred0).numpy()


def train(target, aggregate_loss):
    """Train f_theta and record the LOCATE MAE over training.

    aggregate_loss=True : Clam. `target` is the vector of regional outcomes;
                          the loss compares the regional mean of the predictions to it.
    aggregate_loss=False: baseline. `target` is a ROW x COL matrix of pseudo
                          subregional outcomes; the loss is element-wise.
    """
    set_all_seeds(SEED)
    model = SmallMLP()
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

    X = features(T, C)
    X0 = features(np.zeros_like(T), C)
    target_t = torch.tensor(target, dtype=torch.float32)
    E_true_t = torch.tensor(E_true, dtype=torch.float32)

    mae_history = []
    for epoch in range(N_EPOCHS):
        model.train()
        pred = model(X).reshape(ROW, COL)
        if aggregate_loss:
            loss = torch.mean((pred.mean(dim=1) - target_t) ** 2)
        else:
            loss = torch.mean((pred - target_t) ** 2)
        opt.zero_grad()
        loss.backward()
        opt.step()

        model.eval()
        with torch.no_grad():
            E_pred = (model(X) - model(X0)).reshape(ROW, COL)
            mae_history.append(torch.mean(torch.abs(E_pred - E_true_t)).item())

    return model, mae_history


## Bicubic disaggregation of the regional outcomes

In [ ]:
def to_grid(M):
    """Rearrange a (ROW, COL) region-by-subregion matrix into its (40, 40) spatial map."""
    R, S = int(np.sqrt(ROW)), int(np.sqrt(COL))
    grid = np.zeros((R * S, R * S))
    for i in range(ROW):
        for j in range(COL):
            grid[(i // R) * S + (j // S), (i % R) * S + (j % S)] = M[i, j]
    return grid


def from_grid(grid):
    """Inverse of to_grid: (40, 40) spatial map back to a (ROW, COL) matrix."""
    R, S = int(np.sqrt(ROW)), int(np.sqrt(COL))
    M = np.zeros((ROW, COL))
    for i in range(ROW):
        for j in range(COL):
            M[i, j] = grid[(i // R) * S + (j // S), (i % R) * S + (j % S)]
    return M


from scipy.ndimage import zoom

Y_coarse = Y_agg.reshape(10, 10)     # one pixel per region
Y_up = zoom(Y_coarse, 4, order=3)    # bicubic upsampling to the 40 x 40 subregion grid
Y_bicubic = from_grid(Y_up)          # back to (ROW, COL) indexing, used as training target

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, img, title in zip(axes,
                          [Y_coarse, Y_up, to_grid(Y)],
                          ["Observed regional outcomes", "Bicubic upsampling", "True subregional outcomes"]):
    im = ax.imshow(img, cmap="viridis")
    ax.set_title(title)
    ax.axis("off")
    fig.colorbar(im, ax=ax, shrink=0.8)
plt.tight_layout()
plt.show()


## Train Clam and the two baselines

In [ ]:
Y_uniform = np.repeat(Y_agg[:, None], COL, axis=1)  # uniform disaggregation target

model_clam, mae_clam = train(Y_agg, aggregate_loss=True)
model_uniform, mae_uniform = train(Y_uniform, aggregate_loss=False)
model_bicubic, mae_bicubic = train(Y_bicubic, aggregate_loss=False)

print(f"final LOCATE MAE | Clam: {mae_clam[-1]:.4f} | "
      f"uniform: {mae_uniform[-1]:.4f} | bicubic: {mae_bicubic[-1]:.4f}")


## Results

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(mae_clam, label="Clam (aggregate loss)")
ax.plot(mae_uniform, label="Uniform baseline")
ax.plot(mae_bicubic, label="Bicubic baseline")
ax.set_xlabel("Epoch")
ax.set_ylabel("LOCATE MAE")
ax.legend()
plt.tight_layout()
plt.show()

estimates = [E_true,
             locate_matrix(model_clam, T, C),
             locate_matrix(model_uniform, T, C),
             locate_matrix(model_bicubic, T, C)]
titles = ["Ground truth", "Clam", "Uniform baseline", "Bicubic baseline"]

vmin = min(E.min() for E in estimates)
vmax = max(E.max() for E in estimates)
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, E, title in zip(axes, estimates, titles):
    im = ax.imshow(to_grid(E), cmap="coolwarm", vmin=vmin, vmax=vmax)
    ax.set_title(title)
    ax.axis("off")
fig.colorbar(im, ax=axes, shrink=0.8, label="LOCATE")
plt.show()
